In [ ]:
import requests, pandas as pd, time

In [ ]:
airports=pd.read_csv("/content/drive/MyDrive/02_Airports_Cleaned.csv")

In [ ]:
airports.head()

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,LIH,Lihue Airport,Lihue,HI,USA,21.97598,-159.33896
1,LIT,Bill and Hillary Clinton National Airport (Ada...,Little Rock,AR,USA,34.72940,-92.22425
2,LNK,Lincoln Airport (Lincoln Municipal),Lincoln,NE,USA,40.85097,-96.75925
3,LRD,Laredo International Airport,Laredo,TX,USA,27.54374,-99.46154
4,LSE,La Crosse Regional Airport,La Crosse,WI,USA,43.87938,-91.25654


In [ ]:
airports.isnull().sum()

,0
IATA_CODE,0
AIRPORT,0
CITY,0
STATE,0
COUNTRY,0
LATITUDE,0
LONGITUDE,0


In [ ]:
URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY = "temperature_2m,precipitation,snowfall,wind_speed_10m,visibility,cloud_cover"

In [ ]:
def fetch(code, lat, lon, retries=3):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": "2015-01-01", "end_date": "2015-12-31",
        "hourly": HOURLY,
        "timezone": "auto",
    }
    for i in range(retries):
        try:
            r = requests.get(URL, params=params, timeout=60)
            if r.status_code == 429:      # limit aşılıb
                time.sleep(30 * (i + 1))
                continue
            r.raise_for_status()
            df = pd.DataFrame(r.json()["hourly"])
            df.insert(0, "airport", code)
            return df
        except Exception as e:
            print(f"{code}: cəhd {i+1} uğursuz ({e})")
            time.sleep(5)
    return None

frames, failed = [], []
for n, a in enumerate(airports.itertuples(), 1):
    df = fetch(a.IATA_CODE, a.LATITUDE, a.LONGITUDE)
    if df is None:
        failed.append(a.IATA_CODE)
    else:
        frames.append(df)
    print(f"{n}/{len(airports)} {a.IATA_CODE}")
    time.sleep(1)

weather = pd.concat(frames, ignore_index=True)
weather = weather.rename(columns={"time": "datetime"})
weather.to_csv("weather_2015.csv", index=False)

if failed:
    pd.Series(failed).to_csv("failed_airports.csv", index=False, header=False)
    print("Uğursuz olanlar:", failed)

1/322 LIH
2/322 LIT
3/322 LNK
4/322 LRD
5/322 LSE
6/322 LWS
7/322 MAF
8/322 MBS
9/322 MCI
10/322 MCO
11/322 MDT
12/322 MDW
13/322 MEI
14/322 MEM
15/322 MFE
16/322 MFR
17/322 MGM
18/322 MHK
19/322 MHT
20/322 MIA
21/322 MKE
22/322 MKG
23/322 MLB
24/322 MLI
25/322 MLU
26/322 MMH
27/322 MOB
28/322 MOT
29/322 MQT
30/322 MRY
31/322 MSN
32/322 MSO
33/322 MSP
34/322 MSY
35/322 MTJ
36/322 MVY
37/322 MYR
38/322 OAJ
39/322 OAK
40/322 OGG
41/322 OKC
42/322 OMA
43/322 OME
44/322 ONT
45/322 ORD
46/322 ORF
47/322 ORH
48/322 OTH
49/322 OTZ
50/322 PAH
51/322 PBG
52/322 PBI
53/322 PDX
54/322 PHF
55/322 PHL
56/322 PHX
57/322 PIA
58/322 PIB
59/322 PIH
60/322 PIT
61/322 PLN
62/322 PNS
63/322 PPG
64/322 PSC
65/322 PSE
66/322 PSG
67/322 PSP
68/322 PUB
69/322 PVD
70/322 PWM
71/322 RAP
72/322 RDD
73/322 RDM
74/322 RDU
75/322 RHI
76/322 RIC
77/322 RKS
78/322 RNO
79/322 ROA
80/322 ROC
81/322 ROW
82/322 RST
83/322 RSW
84/322 SAF
85/322 SAN
86/322 SAT
87/322 SAV
88/322 SBA
89/322 SBN
90/322 SBP
91/322 SCC
92/322 S

In [ ]:
weather.shape

(2803200, 8)

In [ ]:
weather.head()

,airport,datetime,temperature_2m,precipitation,snowfall,wind_speed_10m,visibility,cloud_cover
0,LIH,2015-01-01T00:00,19.1,0.0,0.0,11.0,None,85
1,LIH,2015-01-01T01:00,19.0,0.0,0.0,10.5,None,85
2,LIH,2015-01-01T02:00,18.7,0.0,0.0,11.0,None,92
3,LIH,2015-01-01T03:00,18.4,0.0,0.0,9.9,None,84
4,LIH,2015-01-01T04:00,18.3,0.0,0.0,9.7,None,74


In [ ]:
from google.colab import files
files.download("weather_2015.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>